# Exp0.2 — Shift-resolved final-hidden firing-rate analysis

Checkpoint-only analysis. This notebook never trains or evaluates models; it reads finalized CSV artifacts produced by the Slurm post-evaluation pipeline. The primary question is whether regularization preferentially suppresses long-τ neurons, especially after the endpoint, or merely reduces firing globally.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path('..').resolve()
ROOT = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_0_2_endpoint_tail_regularization' / 'endpoint_tail_regularization_v1' / 'experiment_0_2_shift_resolved_fire_rate' / 'shift_fire_rate_v1'
summary = pd.read_csv(ROOT / 'shift_fire_rate_summary.csv')
comparison = pd.read_csv(ROOT / 'shift_fire_rate_comparison.csv')
paired = pd.read_csv(ROOT / 'shift_fire_rate_vs_none.csv')
long_tau = pd.read_csv(ROOT / 'shift_fire_rate_long_tau_summary.csv')
long_tau_comparison = pd.read_csv(ROOT / 'shift_fire_rate_long_tau_comparison.csv')
long_tau_paired = pd.read_csv(ROOT / 'shift_fire_rate_long_tau_vs_none.csv')
print('per-shift rows:', len(summary))
display(summary.head())

## 1. Mean firing rate by synaptic shift
This is the direct per-shift measurement. Compare `none` with each regularized profile for valid-region FR and full-tail FR.

In [ ]:
cols = [
    'architecture', 'objective', 'profile', 'shift',
    'valid_firing_rate_hz_mean', 'tail_firing_rate_hz_mean',
    'tail_stage1_firing_rate_hz_mean', 'tail_stage2_firing_rate_hz_mean',
    'tail_stage3_firing_rate_hz_mean',
]
display(comparison[cols].sort_values(['architecture', 'objective', 'shift', 'profile']))

## 2. Paired percentage change versus frozen Exp0.1 none
Negative percentage changes mean lower firing after regularization. `tail_specific_suppression_pp > 0` means the tail FR was reduced more strongly than the valid-region FR for the same architecture/objective/seed/shift.

In [ ]:
view = paired[paired['profile'] != 'none'].copy()
show = [
    'architecture', 'objective', 'profile', 'seed', 'shift', 'tau_syn_ms',
    'pct_change_valid_firing_rate_hz_vs_none',
    'pct_change_tail_firing_rate_hz_vs_none',
    'tail_specific_suppression_pp',
    'tail_stage1_specific_suppression_pp',
    'tail_stage2_specific_suppression_pp',
    'tail_stage3_specific_suppression_pp',
]
display(view[show].sort_values(['architecture', 'objective', 'profile', 'seed', 'shift']))

## 3. Tail firing-rate curves across shifts
Use the same axes to see whether suppression grows with τ.

In [ ]:
ARCHITECTURE = 'short_mid_long'
OBJECTIVE = 'whole_count_ce'
plot_df = comparison[(comparison['architecture'] == ARCHITECTURE) & (comparison['objective'] == OBJECTIVE)].copy()
fig, ax = plt.subplots(figsize=(8, 5))
for profile, frame in plot_df.groupby('profile'):
    frame = frame.sort_values('shift')
    ax.plot(frame['shift'], frame['tail_firing_rate_hz_mean'], marker='o', label=profile)
ax.set_xlabel('Synaptic shift')
ax.set_ylabel('Tail firing rate (Hz/neuron)')
ax.set_title(f'{ARCHITECTURE} / {OBJECTIVE}: tail FR by shift')
ax.legend()
plt.show()

## 4. Endpoint selectivity across shifts
Positive values are the desired behavior: tail firing falls more than valid firing. Values near zero indicate mostly global suppression.

In [ ]:
selectivity = (
    view.groupby(['architecture', 'objective', 'profile', 'shift'], as_index=False)['tail_specific_suppression_pp']
        .mean()
)
display(selectivity.sort_values(['architecture', 'objective', 'profile', 'shift']))

plot_sel = selectivity[(selectivity['architecture'] == ARCHITECTURE) & (selectivity['objective'] == OBJECTIVE)]
fig, ax = plt.subplots(figsize=(8, 5))
for profile, frame in plot_sel.groupby('profile'):
    frame = frame.sort_values('shift')
    ax.plot(frame['shift'], frame['tail_specific_suppression_pp'], marker='o', label=profile)
ax.axhline(0, linewidth=1)
ax.set_xlabel('Synaptic shift')
ax.set_ylabel('Tail-specific suppression (percentage points)')
ax.set_title(f'{ARCHITECTURE} / {OBJECTIVE}: endpoint selectivity by shift')
ax.legend()
plt.show()

## 5. Pooled long-τ view
The pooled long group uses configured final-hidden s6/s7 neurons (shift ≥6, about 1–2 s at 64 Hz). `short_mid` therefore has no pooled long-τ row; its s2-s5 results remain in the per-shift tables.

In [ ]:
display(long_tau_comparison.sort_values(['architecture', 'objective', 'profile']))
display(
    long_tau_paired[long_tau_paired['profile'] != 'none'][[
        'architecture', 'objective', 'profile', 'seed', 'long_shifts',
        'pct_change_valid_firing_rate_hz_vs_none',
        'pct_change_tail_firing_rate_hz_vs_none',
        'tail_specific_suppression_pp',
    ]].sort_values(['architecture', 'objective', 'profile', 'seed'])
)